# 2. Load and inspect a dataset

A query returns catalog rows. The sections below turn one row into a `SpatialData` object that you can analyse. It
starts with what you can learn about a dataset before you download it, because stores range from 2 MB to almost
100 GB, and a full download is often not what you need.

You will learn how to:

- read the attributes of a `Dataset` and list its elements without downloading anything
- check the size of a store and choose what to fetch
- load a whole store, only some elements, or a copy in a folder of your choice
- find and manage the local cache
- look at the tables, shapes and images of the loaded object
- open the dataset on the website and in Vitessce, and spot viewer links that point at the wrong store

The datasets here are small (under 10 MB each for what is downloaded), so the tutorial runs in a minute or two.
To run the plotting cells, install the client with the `viz` extra, as shown in {doc}`00_overview`.

In [ ]:
import os
import re
import tempfile
import urllib.parse
from pathlib import Path

import matplotlib.pyplot as plt
import platformdirs
import spatialdata_plot  # noqa: F401  (adds the .pl accessor to SpatialData)

import sddb
from sddb.catalog import SITE_URL

cat = sddb.Catalog(version="2026-10-28")
work = Path(tempfile.mkdtemp())  # the files this tutorial writes go here

## A `Dataset` and its attributes

`Catalog.dataset` returns one catalog row as a `Dataset`. Each column of the catalog is an attribute, and a
missing value reads as `None`. The dataset below is a Visium section of a human pancreatic tumour, deposited in
GEO.

In [ ]:
ds = cat.dataset("qEP9pd4LlyKeufe20000")
ds

In [ ]:
{
    column: getattr(ds, column)
    for column in ["technology", "organism", "tissue", "condition", "n_obs", "n_features", "size_bytes", "license", "doi", "default_table"]
}

`condition` and `doi` come from the study, and `default_table` names the table that holds the main count matrix.
A cohort is a sequence of `Dataset` objects, so `for ds in cohort` and `cohort[0]` give the same kind of object.

`Catalog.dataset` also accepts the first 16 characters of the uid (the stem) and the `canonical_key`.
{doc}`03_cohorts_and_manifests` explains when to use which.

## Before you download

Store sizes vary by more than four orders of magnitude. `size_bytes` is the size of the whole store, and most of
it is usually images:

In [ ]:
sizes = cat.to_df()["size_bytes"]
print(f"median {sizes.median() / 1e6:.0f} MB, largest {sizes.max() / 1e9:.0f} GB, {int((sizes > 1e9).sum())} stores above 1 GB")
print(f"this dataset: {ds.size_bytes / 1e6:.1f} MB")

Two things tell you what is inside a store without downloading it. `elements()` reads the store's metadata and
returns every element with its shape and data type. The `sdata_repr` column holds the text that `print(sdata)`
gives for the loaded store.

In [ ]:
for name, info in ds.elements().items():
    print(f"{name:42s} {info['shape']} {info['dtype']}")

In [ ]:
print(ds.sdata_repr)

The store has two H&E images (a 2000-pixel "hires" and a 600-pixel "lowres" version, as Space Ranger writes
them), one shapes element with a circle per spot, and one table with 1142 spots and 17943 genes.

## Load the whole store

`to_sdata()` copies the store into the local cache and opens it with `spatialdata.read_zarr`. A download shows a
progress bar. A later call finds the copy and opens it without downloading, so if you ran the overview, the
dataset is already in the cache and no progress bar appears.

In [ ]:
sdata = ds.to_sdata()
sdata

## A first look at the data

The table is an `AnnData` object. `X` holds the counts, `obs` has one row per spot and `var` one row per gene.
In the pancreas table the gene symbols are the `var_names` and the Ensembl ids are in `var["gene_ids"]`. Other sources
store them the other way round, and the difference shows when you combine datasets ({doc}`03_cohorts_and_manifests`).

In [ ]:
table = sdata.tables["table"]
table

In [ ]:
table.var.head(3)

The table annotates the shapes element. `spatialdata_attrs` names the shapes element (`region`) and the `obs`
column that matches each row to a shape (`instance_key`):

In [ ]:
table.uns["spatialdata_attrs"]

In [ ]:
sdata.shapes["GSM8915497_PDAC04"].head(3)

Each spot is a circle with a centre and a radius. The images and shapes are placed in a common space through
coordinate systems. Space Ranger stores the image at two
resolutions, and the coordinate system `..._downscaled_hires` puts the spots on top of the hires image.

The figure shows the H&E image next to two genes: `COL1A1`, which marks the fibrous stroma around a pancreatic
tumour, and `CELA3A`, an enzyme of acinar cells, the normal exocrine pancreas. The colours show log(1 + counts),
stored as a second table so the counts in `table` stay as they are. `get_extent` gives the bounding box of the
spots, which crops the empty slide around the tissue.

In [ ]:
from spatialdata import get_extent

image = "GSM8915497_PDAC04_hires_image"
spots = "GSM8915497_PDAC04"
coordinates = "GSM8915497_PDAC04_downscaled_hires"

log_counts = table.copy()
log_counts.X = log_counts.X.log1p()
sdata.tables["log_counts"] = log_counts
extent = get_extent(sdata, coordinate_system=coordinates, elements=[spots])

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
sdata.pl.render_images(image).pl.show(coordinate_systems=coordinates, ax=axes[0], title="H&E")
for ax, gene in zip(axes[1:], ["COL1A1", "CELA3A"], strict=True):
    (
        sdata.pl.render_images(image)
        .pl.render_shapes(spots, color=gene, table_name="log_counts")
        .pl.show(coordinate_systems=coordinates, ax=ax, title=f"{gene}, log(1 + counts)")
    )
for ax in axes:
    ax.set_xlim(extent["x"][0] - 40, extent["x"][1] + 40)  # 40 pixels of margin around the spots
    ax.set_ylim(extent["y"][1] + 40, extent["y"][0] - 40)
    ax.set_axis_off()
plt.show()

The section has few counts per spot, so most spots are dark in both panels. `COL1A1` is highest along the left
edge of the section and `CELA3A` at the lower tip, so the two genes mark different parts of the tissue.

## Load only some elements

Most of a large store is images. If you only need the counts, `elements=` copies only the elements you name, as
`"kind/name"` paths from `elements()`. The client also copies the shapes or labels that a kept table annotates,
so the table keeps its link to space.

The next dataset is a Visium section from HEST. Its store is 63 MB, almost all of it a full-resolution image of
8192 by 7953 pixels. Its `default_table` is `None`, which happens for some sources, so read the table name from
`elements()`.

In [ ]:
hest = cat.dataset("USx288NLowiqKbV30000")
print(f"{hest.size_bytes / 1e6:.0f} MB, default_table={hest.default_table}")
for name, info in hest.elements().items():
    print(f"{name:30s} {info['shape']}")

In [ ]:
counts_only = hest.to_sdata(elements=["tables/table"])
counts_only

In [ ]:
def size_on_disk(path: Path) -> float:
    """Total size of the files under ``path``, in MB."""
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e6


print(f"copied {size_on_disk(counts_only.path):.1f} MB of {hest.size_bytes / 1e6:.0f} MB")

The copy holds the table and the spot shapes it annotates, and no image. The warning about variable names comes
from the HEST table, where two genes appear twice. {doc}`03_cohorts_and_manifests` shows how `to_anndata` handles
repeated gene names.

## Keep a copy in your own folder

Without `dest`, stores go to the cache. With `dest`, the store goes to `dest/<uid>.zarr`, and `dest/manifest.json`
records what was copied, from where, and whether the copy is whole or partial. Use `dest` for data that a project
depends on, so it does not live in a cache you might clear.

In [ ]:
project = work / "data"
partial = hest.to_sdata(elements=["tables/table"], dest=project)
print(sorted(p.name for p in project.iterdir()))
print((project / "manifest.json").read_text())

The entry has `status: "partial"` and lists the elements. A later `download()` of a cohort with the HEST dataset into
the same folder fetches the whole store and replaces the partial copy. {doc}`03_cohorts_and_manifests` covers
`download()` and its manifest.

## The cache

The cache is a folder called `sddb` in your platform's user cache directory, or the folder in the environment
variable `SDDB_CACHE_DIR`. Set the variable before you import the client to put the cache on a larger disk.

In [ ]:
cache = Path(os.environ.get("SDDB_CACHE_DIR") or platformdirs.user_cache_dir("sddb"))
for folder in sorted(p for p in cache.iterdir() if p.is_dir()):
    print(f"{folder.name:10s} {size_on_disk(folder):8.1f} MB")

- `catalog` holds the catalog files and `citations.bib`, one per release you opened.
- `stores` holds whole stores from `to_sdata()`, one `<uid>.zarr` per dataset.
- `elements` holds the partial copies from `to_sdata(elements=...)`.

The cache is keyed by the full uid. When a dataset gets a new version, its uid changes and the client downloads
the new store rather than reusing the old copy. Nothing is ever deleted for you, so remove the folder by hand when
it gets large. The next call downloads what it needs again.

## The website and Vitessce

Every dataset has a page on the website, at the address built from its `slug`. `card_url` links the dataset in
Lamin, and `dataset_url` links the source (here the GEO series).

In [ ]:
print(f"{SITE_URL}/datasets/{ds.slug}/")
print(ds.card_url)
print(ds.dataset_url)

`viewer_url()` returns a link that opens the dataset in Vitessce, a viewer that runs in the browser. The link
contains the whole viewer configuration, so it is long. `view()` opens it in your browser, and
`view_interactive()` shows it inside a notebook. Both need the `viz` extra.

In [ ]:
url = ds.viewer_url()
print(url[:80] + "...", f"({len(url)} characters)")

In the 2026-10-28 release, 333 of the 5937 viewer links load the store of another dataset. The website checks every link
and hides the 333 that do not match, but `viewer_url()` returns the link as it is in the catalog. The function
below finds the store ids in a viewer link and compares them with the dataset's own store:

In [ ]:
def viewer_matches_store(dataset: sddb.Dataset) -> bool:
    """True when every store in the Vitessce link of ``dataset`` is the dataset's own store."""
    if not dataset.vitessce_url:
        return True
    text, previous = dataset.vitessce_url, None
    while text != previous:  # the configuration is URL-encoded twice
        previous, text = text, urllib.parse.unquote(text)
    linked = {s[:16] for s in re.findall(r"\.lamindb/(\w{16,20})\.zarr", text)}
    return linked <= {dataset.uid[:16]}


mismatched = [d.uid for d in cat.query() if not viewer_matches_store(d)]
print(len(mismatched), "viewer links point at another store")
print("this dataset:", viewer_matches_store(ds))

For example, the viewer link of a small Xenium kidney section, `NHyd5DFysygNrkdu0000`, opens a different
dataset. Run the check before you share a viewer link or use one to look at a dataset.

In [ ]:
print("NHyd5DFysygNrkdu0000" in mismatched)

## Summary

- `Dataset` attributes, `elements()` and `sdata_repr` describe a dataset without a download. Check `size_bytes`
  first.
- `to_sdata()` caches the whole store, `elements=` copies only what you name, and `dest=` keeps a copy with a
  manifest in a folder of your choice.
- The cache lives in `SDDB_CACHE_DIR` or the user cache directory, keyed by full uid.
- Check a Vitessce link with the function above before you use it.

Next: [3. Cohorts and manifests](03_cohorts_and_manifests.ipynb), which works with many datasets at once and
makes the selection reproducible.